# NBFNet — Bellicum

Solo NBFNet y solo el grafo de Bellicum, el más grande de los tres (120 cláusulas, 631 enunciados). Sus partes **no están verificadas a mano**: en el
paso 5, «la parte correcta» es la que puso la extracción. Mismos pasos y mismas pruebas que
[`transe_transh_transr/Bellicum.ipynb`](../transe_transh_transr/Bellicum.ipynb), pero NBFNet es otra
clase de modelo: no guarda un vector por entidad, razona con los caminos del grafo.

1. el grafo como tripletas `(h, r, t)`;
2. qué hace NBFNet;
3. entrenarlo y ver qué ha aprendido;
4. ¿adivina aristas que no ha visto?;
5. la pregunta que nos importa: ¿adivina la parte de un enunciado?, y por qué no.

In [ ]:
import collections
import sys
import time
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# link_prediction.py está en notebooks/, una carpeta por encima de este cuaderno.
for folder in (Path.cwd(), Path.cwd().parent, Path.cwd() / "notebooks"):
    if (folder / "link_prediction.py").exists():
        sys.path.insert(0, str(folder))
        break
import link_prediction as lp


# Configuración: la de T1 en link_prediction.ipynb (sección 4), la misma en los tres contratos.
# NBFNet no entró en el ajuste por rejilla de la sección 3: son los valores por defecto de
# lp.train_nbfnet, salvo las épocas.
#   epochs      cuántas veces se recorren todas las preguntas de entrenamiento
#   dim         números en el vector de cada nodo mientras se responde una pregunta
#   layers      rondas de propagación: el largo máximo de los caminos que mira
#   lr          tasa de aprendizaje
#   batch_size  preguntas por corrección
# El optimizador es Adam. Esta implementación mínima corre en CPU.
CONFIG = {"epochs": 10, "dim": 32, "layers": 4, "lr": 5e-3, "batch_size": 16}
# Veces que se repite el paso 5 con repartos distintos. 1, como link_prediction.ipynb hizo con
# este contrato; con 3, el paso 5 tarda el triple.
REPEATS = 1

contract_name = "Bellicum"
contract = lp.load_contract(contract_name)
e2i, r2i = lp.entity_ids(contract)
i2e = {i: e for e, i in e2i.items()}

In [ ]:
ALL_TYPES = ("is_part_of", "assigns_obligation_to", "grants_right_to", "defines", "uses", "references", "depends_on", "supersedes", "modifies")
print(f"contrato {contract_name}")
print(f"{len(contract.triples)} tripletas, {len(e2i)} entidades, {len(r2i)} tipos de arista presentes")
print("tipos de arista que no aparecen:", ", ".join(r for r in ALL_TYPES if r not in r2i) or "ninguno")
lp.table([{"tipo de arista": r, "tripletas": n} for r, n in collections.Counter(r for _, r, _ in contract.triples).most_common()])

Una tripleta de cada tipo de arista, con el texto de cada nodo para poder leerlas:

In [ ]:
def label(node):
    if node in contract.parties:
        return f"{node} ({contract.parties[node]['name']})"
    if node in contract.clauses:
        c = contract.clauses[node]
        return f"{node} (§{c['ref']} {c.get('heading') or ''})".strip()
    if node in contract.statements:
        return f"{node} «{lp.statement_text(contract.statements[node])[:55]}…»"
    return node


examples = [next(t for t in contract.triples if t[1] == r) for r in sorted(r2i)]
lp.table([{"h": label(h), "r": r, "t": label(t)} for h, r, t in examples])

## 2. Qué hace NBFNet

Los modelos de las otras carpetas guardan un vector por entidad. NBFNet no guarda ninguno. Para
una pregunta `(h, r, ?)`:

1. pone en `h` un vector que depende solo del tipo de arista preguntado, `r`; los demás nodos
   empiezan en cero;
2. en cada ronda, cada nodo recibe un mensaje de cada vecino, por las aristas en los dos sentidos:
   el vector del vecino multiplicado, número a número, por un vector propio del tipo de esa arista
   y de su sentido. Esos vectores de arista también dependen de `r`;
3. tras cuatro rondas, el vector de cada nodo `t` resume los caminos de hasta cuatro aristas que
   van de `h` a `t`;
4. una red pequeña convierte ese vector en la puntuación de `t`.

Es el algoritmo de Bellman-Ford (el de los caminos más cortos) con la suma y el mínimo cambiados
por operaciones que se aprenden. Dos consecuencias:

- **Es inductivo.** Lo que aprende son los vectores de los tipos de arista, no los de las
  entidades: vale para nodos que nunca vio y, en principio, para otro contrato con las mismas
  aristas.
- **Se explica con caminos.** La puntuación de `t` sale de los caminos entre `h` y `t`, y esos
  caminos se pueden mostrar.

Es una implementación mínima (`_nbfnet_class` en `link_prediction.py`), sin las optimizaciones del
artículo (Zhu et al., NeurIPS 2021).

**El muchos-a-uno no existe aquí**: no hay vectores de enunciado que amontonar junto a su parte.
Lo que cuenta es qué caminos unen un enunciado con cada parte. Cuántas aristas de parte llegan a
cada una:

In [ ]:
counts = collections.Counter((r, t) for _, r, t in contract.triples if r in lp.PARTY_RELATIONS)
lp.table([{"tipo de arista": r, "parte": label(p), "aristas": n} for (r, p), n in sorted(counts.items())])

Cada una de esas aristas es un camino de un paso hasta su parte. Si se esconde la de un enunciado,
el camino más corto que le queda suele ser de tres: enunciado → su cláusula → otro enunciado de la
cláusula → la parte de ese otro. Con cuatro rondas, NBFNet los ve todos: tiene a mano justo lo que
usa la mayoría de su cláusula.

## 3. Entrenar y ver qué ha aprendido

**Entrenar** NBFNet es hacerle preguntas `(h, r, ?)` cuya respuesta se conoce y corregir sus
vectores de arista y su red para que la respuesta verdadera quede arriba entre los candidatos.
Aquí las preguntas son todas las tripletas del grafo, y los candidatos, todas las entidades.
Mientras responde una pregunta, su propia arista se quita del grafo: si no, aprendería a ver la
respuesta en vez de a deducirla. Se repite diez veces sobre todas las preguntas (las *épocas*).
Corre en CPU: en SteelVault tarda segundos; en Bellicum, unos diez minutos, y el paso 4 entrena
otro igual.

In [ ]:
entities = list(e2i)
start = time.time()
model = lp.train_nbfnet(contract.triples, contract.triples, lambda h, r: entities, e2i, r2i, seed=0, **CONFIG)
lp.table([{"modelo": "NBFNet", **CONFIG, "preguntas": len(contract.triples), "segundos": round(time.time() - start, 1)}])

**Cómo se lee el modelo.**

Se le hace una pregunta: ¿a quién obliga una obligación concreta? La celda toma la primera
obligación del contrato que obliga a una sola parte. El modelo puntúa todas las entidades y las
ordena; la posición de la respuesta correcta es su **rango**: 1 si sale la primera.

NBFNet no tiene dónde memorizar la respuesta: no guarda nada por entidad. Al entrenar, esta
pregunta se hizo sin su arista; ahora la arista está en el grafo, como un camino de un paso hasta
la respuesta, y el modelo decide cuánto pesa.

In [ ]:
example = next(sid for sid, r, _ in lp.party_queries(contract) if r == "assigns_obligation_to" and sid not in contract.twin and contract.statements[sid]["kind"] == "obligation")
answer = contract.party_edge(example)[1]
display(Markdown(f"Pregunta: `({example}, assigns_obligation_to, ?)`, «{lp.statement_text(contract.statements[example])[:100]}». Respuesta correcta: `{answer}` ({contract.parties[answer]['name']})."))


def top(model, h, r, k=5):
    scores = lp.nbfnet_scores(model, [(h, r)], e2i, r2i)[0]
    order = np.argsort(-scores)
    return [f"{i2e[i]} ({scores[i]:.2f})" for i in order[:k]]


lp.table([{"modelo": "NBFNet", **{f"{k + 1}.º": x for k, x in enumerate(top(model, example, "assigns_obligation_to"))}}])

Tiene el formato Entidad (puntuación). Solo importa el orden: la escala no tiene un significado fijo.

## 4. ¿Adivina aristas que no ha visto?

Ahora la prueba de verdad: se esconde un 10% de las tripletas al azar, se entrena con el resto y
se pregunta por las escondidas. Para resumir todos esos rangos en un número se usan dos medidas:

- **MRR** (*rango recíproco medio*): la media de `1 / rango`. Si la respuesta sale 1.ª, 2.ª y
  4.ª en tres preguntas, el MRR es `(1 + 1/2 + 1/4) / 3 = 0,58`. Vale 1 si siempre sale la primera.
- **Hits@k**: la fracción de preguntas en que la respuesta sale entre las *k* primeras.

El rango es **filtrado**: si una pregunta tiene varias respuestas verdaderas conocidas (una
cláusula tiene muchos enunciados), las otras no cuentan como error. Como referencia, el azar:
ordenar todas las entidades sin mirar nada. Un empate cuenta como el rango medio de los
empatados. En NBFNet puede haberlos: todos los nodos a los que no llega ningún camino de hasta
cuatro aristas desde `h` reciben la misma puntuación.

In [ ]:
rng = np.random.default_rng(0)
hidden_idx = set(rng.choice(len(contract.triples), size=len(contract.triples) // 10, replace=False).tolist())
train = [t for i, t in enumerate(contract.triples) if i not in hidden_idx]
hidden = [contract.triples[i] for i in sorted(hidden_idx)]
true_tails = collections.defaultdict(set)
for h, r, t in contract.triples:
    true_tails[(h, r)].add(e2i[t])


def filtered_ranks(scores, triples):
    """Rango filtrado de la respuesta. Un empate cuenta como el rango medio de los empatados."""
    ranks = []
    for row, (h, r, t) in zip(scores, triples):
        row = row.copy()
        row[[i for i in true_tails[(h, r)] if i != e2i[t]]] = -np.inf
        target = row[e2i[t]]
        ranks.append(1 + np.sum(row > target) + 0.5 * (np.sum(row == target) - 1))
    return np.array(ranks)


start = time.time()
held_out = lp.train_nbfnet(train, train, lambda h, r: entities, e2i, r2i, seed=0, **CONFIG)
ranks = filtered_ranks(lp.nbfnet_scores(held_out, [(h, r) for h, r, _ in hidden], e2i, r2i), hidden)
n = len(e2i)
display(Markdown(f"{len(hidden)} tripletas escondidas de {len(contract.triples)}; entrenado en {time.time() - start:.0f} s"))
lp.table([
    {"modelo": "al azar (esperado)", "MRR": float(np.mean([1 / k for k in range(1, n + 1)])), "Hits@1": 1 / n, "Hits@10": 10 / n},
    {"modelo": "NBFNet", "MRR": float(np.mean(1 / ranks)), "Hits@1": float(np.mean(ranks == 1)), "Hits@10": float(np.mean(ranks <= 10))},
])

Como referencia, en el ajuste de [`link_prediction.ipynb`](../link_prediction.ipynb) el mejor
modelo de vectores, TransE, dio un MRR de 0,46 en SteelVault (otra muestra del 10%).

Las escondidas son muchas para mirarlas una a una. Primero, el MRR según el tipo de arista
escondida:

In [ ]:
by_type = collections.defaultdict(list)
for k, (_, r, _) in enumerate(hidden):
    by_type[r].append(k)
lp.table([
    {"tipo de arista": r, "escondidas": len(ks), "MRR": float(np.mean(1 / ranks[ks])), "Hits@1": float(np.mean(ranks[ks] == 1))}
    for r, ks in sorted(by_type.items(), key=lambda item: -len(item[1]))
])

Y las quince primeras tripletas escondidas, con el rango de NBFNet:

In [ ]:
degree = collections.Counter(x for h, _, t in train for x in (h, t))
lp.table([{"h": h, "r": r, "t": t, "rango": float(ranks[k]), "aristas de h que quedan": degree[h]} for k, (h, r, t) in enumerate(hidden[:15])], digits=1)

Cómo leerlas: un 1 es un acierto a la primera; un número alto, que la respuesta quedó muy abajo
entre todas las entidades. Fíjate en qué tipos de arista (`r`) dan rangos bajos y en cuáles altos,
y en si el origen `h` tiene otras aristas que el modelo haya podido ver.

La última columna importa aquí. Si al origen `h` no le queda ninguna arista, NBFNet no tiene ningún
camino por el que empezar: todas las entidades empatan y el rango es el del medio. Lo mismo para un
destino al que no llega ningún camino de hasta cuatro aristas.

## 5. La pregunta que nos importa: ¿qué parte?

Se esconde la arista de parte de un enunciado y se le pregunta a NBFNet cuál de las **dos** partes
encaja mejor. Se mide el **acierto**: la fracción de enunciados con la parte bien. Un empate entre
las dos cuenta medio acierto.

- Solo los **enunciados unilaterales** (la celda dice cuántos). Los gemelos —la misma frase
  emitida una vez por parte, como en *«each party shall…»*— tienen las dos partes por
  construcción y no hay nada que adivinar.
- Se reparten los unilaterales en cinco grupos; se esconde un grupo, se entrena con el resto del
  grafo y se pregunta por ese grupo. Así hasta pasar por los cinco, y todo `REPEATS` veces con
  repartos distintos.
- Aquí NBFNet se entrena **solo con preguntas de parte**: las aristas de parte conocidas de los
  unilaterales, con las dos partes como únicas candidatas, como en `link_prediction.ipynb`. Los
  gemelos siguen en el grafo pero no se usan como preguntas: le enseñarían una regla falsa, «la
  parte contraria a la de mi vecino idéntico» ([`link_prediction.ipynb`](../link_prediction.ipynb),
  sección 4, lo mide).
- Dos referencias sin modelo: **siempre la parte más frecuente** y **la mayoría de su cláusula**
  (la parte que más aparece entre los otros enunciados de la misma cláusula).
- La parte «correcta» es la que puso la extracción: en este contrato nadie la ha revisado a mano.

In [ ]:
queries = [q for q in lp.party_queries(contract) if q[0] not in contract.twin]
twins = [q for q in lp.party_queries(contract) if q[0] in contract.twin]
print(f"{len(queries)} enunciados unilaterales, {len(twins)} gemelos")
parties = sorted(contract.parties)
correct = collections.defaultdict(list)  # método -> [acierto de cada repetición]
predictions = collections.defaultdict(dict)  # método -> {enunciado: parte o None} de la primera repetición


def pick(row):
    """La parte con más puntuación; None si las dos empatan."""
    scores = [row[e2i[p]] for p in parties]
    return None if scores[0] == scores[1] else parties[int(np.argmax(scores))]


start = time.time()
for rep in range(REPEATS):
    fold_of = lp.folds(len(queries), 5, seed=rep)
    hits = collections.Counter()
    for fold in range(5):
        test = [q for q, f in zip(queries, fold_of) if f == fold]
        known = [q for q, f in zip(queries, fold_of) if f != fold] + twins
        graph = [t for t in contract.triples if t not in set(test)]
        base = lp.t1_baselines(contract, known, test)
        guesses = {"siempre la más frecuente": base["mayoría"], "mayoría de su cláusula": base["cláusula"]}
        training = [q for q in known if q[0] not in contract.twin]
        nbf = lp.train_nbfnet(graph, training, lambda h, r: parties, e2i, r2i, seed=rep, **CONFIG)
        scores = lp.nbfnet_scores(nbf, [(h, r) for h, r, _ in test], e2i, r2i)
        guesses["NBFNet"] = {sid: pick(row) for (sid, _, _), row in zip(test, scores)}
        for method, guess in guesses.items():
            hits[method] += sum(1.0 if guess[sid] == p else 0.5 if guess[sid] is None else 0.0 for sid, _, p in test)
            if rep == 0:
                predictions[method].update(guess)
    for method, k in hits.items():
        correct[method].append(k / len(queries))
print(f"{REPEATS * 5} entrenamientos en {time.time() - start:.0f} s")
print("empates de NBFNet en la primera repetición:", sum(v is None for v in predictions["NBFNet"].values()))
lp.table(sorted([{"método": m, "acierto": float(np.mean(v)), "peor repetición": min(v), "mejor repetición": max(v)} for m, v in correct.items()], key=lambda r: -r["acierto"]))

En [`link_prediction.ipynb`](../link_prediction.ipynb), con este mismo reparto, NBFNet acertó el
68%: el mejor modelo de grafo en este contrato y algo por encima de la mayoría de su cláusula
(65%). Pero la regla del sujeto (la primera parte que nombra la frase) llega al 76%, y una
regresión logística sobre el texto del enunciado, al 75%. Esta tabla debería repetirlo.

### Por qué

NBFNet mira caminos. Al esconder la arista de parte de un enunciado, ¿qué caminos le quedan hasta
cada parte? La celda cuenta los recorridos de una a cuatro aristas —las cuatro rondas del
modelo—, en cualquier sentido, que salen del enunciado y acaban en cada parte, sin su arista de
parte (en el paso 5 faltaban también las de los otros enunciados de su grupo; aquí solo la suya).
Luego mira cuántas veces la parte con más recorridos es la correcta, y cuántas coincide con la
mayoría de su cláusula:

In [ ]:
n = len(e2i)
A = np.zeros((n, n))
for h, r, t in contract.triples:
    A[e2i[h], e2i[t]] += 1
    A[e2i[t], e2i[h]] += 1


def walks_to_parties(sid, steps=CONFIG["layers"]):
    """Recorridos de 1 a `steps` aristas, en cualquier sentido, del enunciado a cada parte, sin
    sus propias aristas de parte."""
    s = e2i[sid]
    own = [e2i[t] for h, r, t in contract.triples if h == sid and r in lp.PARTY_RELATIONS]
    for p in own:
        A[s, p] -= 1
        A[p, s] -= 1
    v, total = np.zeros(n), np.zeros(n)
    v[s] = 1
    for _ in range(steps):
        v = A @ v
        total += v
    for p in own:
        A[s, p] += 1
        A[p, s] += 1
    return {p: total[e2i[p]] for p in parties}


def score(guess, truth):
    return 1.0 if guess == truth else 0.5 if guess is None else 0.0


walks = {sid: walks_to_parties(sid) for sid, _, _ in queries}
most = {sid: None if w[parties[0]] == w[parties[1]] else max(w, key=w.get) for sid, w in walks.items()}
truth = {sid: p for sid, _, p in queries}
clause = predictions["mayoría de su cláusula"]
display(Markdown(
    f"La parte con más recorridos coincide con la mayoría de su cláusula en {sum(most[s] == clause[s] for s in truth)} "
    f"de {len(truth)} enunciados. Sin ningún recorrido hasta ninguna parte: {sum(max(w.values()) == 0 for w in walks.values())}."
))
lp.table([
    {"regla": "la parte con más recorridos", "acierto": float(np.mean([score(most[s], truth[s]) for s in truth]))},
    {"regla": "mayoría de su cláusula", "acierto": float(np.mean([score(clause[s], truth[s]) for s in truth]))},
    {"regla": "NBFNet", "acierto": float(np.mean([score(predictions["NBFNet"][s], truth[s]) for s in truth]))},
])

Si la parte con más recorridos acierta más o menos lo mismo que la mayoría de su cláusula y
coincide con ella casi siempre, es porque los recorridos cortos hasta una parte pasan casi todos
por la cláusula: enunciado → cláusula → otro enunciado → su parte. Es la misma información que la
mayoría de la cláusula, contada de otra forma. NBFNet aprende cuánto pesa cada clase de camino,
pero no tiene otros: **ningún camino dice quién carga con el enunciado.**

Lo que dice quién carga está **en el texto**, no en el grafo. Unos casos en que NBFNet falla, con
los recorridos que le quedan hasta cada parte:

In [ ]:
fails = [(sid, p) for sid, _, p in queries if predictions["NBFNet"][sid] != p][:6]
lp.table([{
    "enunciado": sid,
    "texto": lp.statement_text(contract.statements[sid])[:80],
    "recorridos hasta cada parte": ", ".join(f"{contract.parties[q]['name'][:22]} {walks[sid][q]:.0f}" for q in parties),
    "NBFNet dice": "empate" if predictions["NBFNet"][sid] is None else contract.parties[predictions["NBFNet"][sid]]["name"][:22],
    "correcta": contract.parties[p]["name"][:22],
} for sid, p in fails])

## En resumen

- El grafo de Bellicum son las tripletas `(h, r, t)` del paso 1.
- NBFNet no guarda vectores de entidad: responde con los caminos que unen el origen de la pregunta
  con cada candidato. Por eso no tiene el problema del muchos-a-uno, y por eso cada respuesta se
  puede explicar con caminos.
- El paso 4 dice cuánto deduce del grafo en general, y en qué tipos de arista.
- El paso 5 dice si adivina la parte de un enunciado mejor que mirar su cláusula. El apartado
  *Por qué* cuenta los caminos que tiene a mano: si casi todos pasan por la cláusula, no puede
  saber más que la mayoría de la cláusula.
- Donde NBFNet sí aportó algo en `link_prediction.ipynb` fue en las relaciones entre cláusulas
  (T3), puesto detrás de la regla del número citado y sobre aristas que nadie ha revisado.